# CPU performance basics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shoaibphysics/custom-kernels/blob/main/notebooks/02_cpu_performance_basics.ipynb)

How do we measure a computation and explain its performance?

We move from analytical estimates to experiments on the CPU.
Through small examples, we explore timing, execution overhead,
vectorization, and memory access patterns.

We begin with the same vector addition studied in [01_performance_basics.ipynb](01_performance_basics.ipynb),
this time using actual arrays.

## 1. From a model to an actual computation

In [01_performance_basics.ipynb](01_performance_basics.ipynb),
we calculated how much work vector addition would require.
We now allocate the arrays and perform that computation:

$$
z_i = x_i + y_i.
$$

We use NumPy arrays with dtype float32, so each element occupies
4 bytes. For a simple correctness check, every element of x is 2
and every element of y is 3. Every output should therefore be 5.

The expression `z = x + y` allocates a new output array and
computes the result. Instead, we create the output storage first:

`z = np.empty_like(x)`

Then we write the addition results into that existing array:

`np.add(x, y, out=z)`

When timing, we can place the allocation before the timer and
measure the addition separately. This excludes creating a new
output array from the timed operation. Reading the inputs and
writing the output still remain part of the computation.

`np.empty_like(x)` creates an array with the same shape and dtype
as x, but leaves its values uninitialized. We must write the
result before reading it.

These ordinary NumPy arrays and operations use the CPU.

In [1]:
import platform  # Record the operating system and architecture for context.
import numpy as np

print("Platform:", platform.system(), platform.machine())
print("NumPy version:", np.__version__)

n_elements = 1_000_000

x = np.full(n_elements, 2.0, dtype=np.float32)
y = np.full(n_elements, 3.0, dtype=np.float32)
z = np.empty_like(x)  # Allocate output storage before the addition.

# Perform the addition, writing into the existing output array.
np.add(x, y, out=z)

# These particular inputs have an exactly representable result.
assert np.all(z == 5.0)

# Logical traffic for the addition alone:
# read x once, read y once, and write z once.
logical_bytes = x.nbytes + y.nbytes + z.nbytes

print("First five outputs:", z[:5])
print("Output dtype:", z.dtype)
print("Bytes per element:", z.itemsize)
print(f"Logical addition traffic: {logical_bytes / 1e6:.2f} MB")
print("Correctness check passed.")

Platform: Darwin arm64
NumPy version: 2.5.3
First five outputs: [5. 5. 5. 5. 5.]
Output dtype: float32
Bytes per element: 4
Logical addition traffic: 12.00 MB
Correctness check passed.


## 2. Measuring execution time

We use `time.perf_counter()` to read a high-resolution clock
before and after the operation:

$$
t_{\mathrm{elapsed}} = t_{\mathrm{end}} - t_{\mathrm{start}}.
$$

The difference is measured in seconds. It represents elapsed
wall-clock time, including any interruptions while the operation
is running.

Our NumPy CPU addition finishes before the function returns,
so no GPU synchronization is needed here.

The arrays already exist. We time the addition call, excluding
array creation, correctness checks, and printing. Function-call
overhead and the input reads and output writes remain included.

We first perform an untimed warmup call using the same arrays.
This prepares the operation and buffers for measurement, but
does not guarantee identical timings across runs.

We then calculate:

$$
B_{\mathrm{effective}} = \frac{Q}{t_{\mathrm{elapsed}}},
\qquad
P_{\mathrm{achieved}} = \frac{F}{t_{\mathrm{elapsed}}}.
$$

A single measurement is an initial observation. In the next
section, we will repeat the measurement and examine its variation.

In [2]:
from time import perf_counter  # High-resolution elapsed-time measurement.

# Untimed warmup using the existing arrays.
np.add(x, y, out=z)

start = perf_counter()
np.add(x, y, out=z)
end = perf_counter()

elapsed_seconds = end - start

# Check correctness outside the timed region.
assert np.all(z == 5.0)

logical_bytes = x.nbytes + y.nbytes + z.nbytes
total_flops = x.size

effective_bandwidth = logical_bytes / elapsed_seconds
arithmetic_throughput = total_flops / elapsed_seconds

print("Measured NumPy CPU addition — one timed call")
print(f"Elapsed time: {elapsed_seconds * 1e6:.2f} microseconds")
print(f"Effective bandwidth: {effective_bandwidth / 1e9:.2f} GB/s")
print(f"Arithmetic throughput: {arithmetic_throughput / 1e9:.2f} GFLOP/s")
print("Correctness check passed.")

Measured NumPy CPU addition — one timed call
Elapsed time: 122.25 microseconds
Effective bandwidth: 98.16 GB/s
Arithmetic throughput: 8.18 GFLOP/s
Correctness check passed.


## 3. Repeated measurements

Execution time can vary between calls because of system activity,
CPU scheduling, cache state, and other effects. A single timing
does not show this variation.

We repeat the same operation and record each elapsed time.
The input values, array sizes, dtype, and output buffer remain
unchanged.

We summarize the measurements using:

- Minimum: the fastest observed call.
- Median: the middle of the timing distribution.
- Maximum: the slowest observed call.

The median is less sensitive than the mean to a few unusually
slow calls, although it does not eliminate measurement bias.

Warmup calls happen before measurement. Correctness checks and
printing happen afterward.

Because we reuse the same arrays, these measurements describe
repeated execution with potentially cached data. They are not
a direct measurement of main-memory bandwidth.

In [3]:
from statistics import median

# Untimed warmup.
for _ in range(5):
    np.add(x, y, out=z)

elapsed_samples = []

for _ in range(50):
    start = perf_counter()
    np.add(x, y, out=z)
    end = perf_counter()

    elapsed_samples.append(end - start)

# Correctness check outside all timed regions.
assert np.all(z == 5.0)

median_seconds = median(elapsed_samples)
logical_bytes = x.nbytes + y.nbytes + z.nbytes
bandwidth_at_median = logical_bytes / median_seconds

print(f"Timed calls: {len(elapsed_samples)}")
print(f"Minimum time: {min(elapsed_samples) * 1e6:.2f} microseconds")
print(f"Median time: {median_seconds * 1e6:.2f} microseconds")
print(f"Maximum time: {max(elapsed_samples) * 1e6:.2f} microseconds")
print(f"Effective bandwidth at median time: {bandwidth_at_median / 1e9:.2f} GB/s")

Timed calls: 50
Minimum time: 86.42 microseconds
Median time: 99.33 microseconds
Maximum time: 218.46 microseconds
Effective bandwidth at median time: 120.81 GB/s


## 4. Python loops and compiled array operations

We can express the same addition in two ways:

- A Python loop that reads, adds, and writes one element per iteration.
- A NumPy call that processes the arrays using compiled code.

Both implementations compute the same mathematical result.
However, the Python loop repeatedly performs interpreter-level
work for iteration, indexing, scalar operations, and assignment.

NumPy moves the elementwise loop into compiled code, avoiding
much of that per-element Python overhead.

We compare both implementations using the same FP32 inputs
and an existing output array. Allocation, correctness checks,
and printing remain outside the timed regions.

We use 100,000 elements to keep the Python-loop experiment short.

This comparison does not isolate the benefit of SIMD instructions
or multiple CPU cores. Several implementation differences contribute
to the timing difference.

In [4]:
a = np.full(100_000, 2.0, dtype=np.float32)
b = np.full(100_000, 3.0, dtype=np.float32)
out = np.empty_like(a)

def add_python_loop(a, b, out):
    for i in range(a.size):
        out[i] = a[i] + b[i]

implementations = [("Python loop", add_python_loop),("NumPy", np.add)]

median_times = {}

for name, operation in implementations:
    # Untimed warmup.
    operation(a, b, out=out)

    samples = []
    for _ in range(10):
        start = perf_counter()
        operation(a, b, out=out)
        end = perf_counter()

        samples.append(end - start)

    # Check each implementation outside the timed region.
    assert np.all(out == 5.0)

    median_times[name] = median(samples)
    print(f"{name}: {median_times[name] * 1e6:.2f} microseconds")

ratio = median_times["Python loop"] / median_times["NumPy"]
print(f"Python-loop / NumPy median-time ratio: {ratio:.2f}x")

Python loop: 8256.48 microseconds
NumPy: 6.63 microseconds
Python-loop / NumPy median-time ratio: 1246.25x


## 5. Memory layout and strides

Performance depends partly on how we access data in memory.

In a contiguous one-dimensional array, consecutive elements
sit next to each other. For float32, each element occupies
4 bytes, so consecutive elements start 4 bytes apart.

An array's stride tells us the distance, in bytes, between
the starts of consecutive elements along an axis.

Consider selecting every second element:

`strided = storage[::2]`

This creates a view of the original storage without copying
the selected values. Each element still occupies 4 bytes,
but consecutive selected elements start 8 bytes apart.

Calling `strided.copy()` creates a new contiguous array
containing the same values, now packed next to each other.

The arrays therefore have the same values, element count,
and dtype, but different memory layouts.

CPUs transfer data through caches in blocks called cache lines.
Accessing nearby elements can make better use of each block.
We will inspect the layout first, then benchmark its effect.

In [5]:
storage = np.arange(16, dtype=np.float32)

strided = storage[::2]       # Select every second element without copying.
contiguous = strided.copy()  # Copy those values into consecutive storage.

assert np.array_equal(strided, contiguous)

for name, array in [("Strided view", strided),
                    ("Contiguous copy", contiguous)]:
    print(f"\n{name}")
    print("Values:", array)
    print("Bytes per element:", array.itemsize)
    print("Stride in bytes:", array.strides[0])
    print("Logical data size in bytes:", array.nbytes)

    # Starting byte positions, relative to this array's first element.
    offsets = np.arange(array.size) * array.strides[0]
    print("Element byte offsets:", offsets)


Strided view
Values: [ 0.  2.  4.  6.  8. 10. 12. 14.]
Bytes per element: 4
Stride in bytes: 8
Logical data size in bytes: 32
Element byte offsets: [ 0  8 16 24 32 40 48 56]

Contiguous copy
Values: [ 0.  2.  4.  6.  8. 10. 12. 14.]
Bytes per element: 4
Stride in bytes: 4
Logical data size in bytes: 32
Element byte offsets: [ 0  4  8 12 16 20 24 28]


## 6. Benchmarking contiguous and strided inputs

We compare two ways of storing the inputs to:

$$
z_i = x_i + y_i.
$$

- Contiguous inputs: consecutive elements start 4 bytes apart.
- Strided inputs: consecutive selected elements start 8 bytes apart.

We create the strided views first, then copy their values into
contiguous arrays. Both cases therefore have identical values,
element counts, and float32 dtypes.

Both write into the same preallocated, contiguous output array.
Only the input layout changes.

Each case performs N additions. Its logical traffic is:

$$
Q = 4N + 4N + 4N = 12N \text{ bytes}.
$$

This counts the selected input values and the output values.
Actual transfers through the memory hierarchy can differ because
hardware moves data in blocks that may include unused neighbors.

We warm up each case, measure 50 calls, and report the median.
Allocation, copying, and correctness checks stay outside timing.

These measurements reuse the same buffers and may benefit from
caches. The calculated effective bandwidth is not a direct
measurement of main-memory bandwidth.

In [6]:
n_elements = 1_000_000

# Each backing array has twice as many elements as we will select.
storage_x = np.full(2 * n_elements, 2.0, dtype=np.float32)
storage_y = np.full(2 * n_elements, 3.0, dtype=np.float32)

# Select one million elements, with gaps between them.
strided_x = storage_x[::2]
strided_y = storage_y[::2]

# Copy the same values into contiguous storage.
contiguous_x = strided_x.copy()
contiguous_y = strided_y.copy()

result = np.empty(n_elements, dtype=np.float32)

cases = [
    ("Contiguous", contiguous_x, contiguous_y),
    ("Strided", strided_x, strided_y),
]

for name, left, right in cases:
    # Untimed warmup.
    for _ in range(5):
        np.add(left, right, out=result)

    samples = []
    for _ in range(50):
        start = perf_counter()
        np.add(left, right, out=result)
        end = perf_counter()
        samples.append(end - start)

    assert np.all(result == 5.0)

    median_seconds = median(samples)
    logical_bytes = left.nbytes + right.nbytes + result.nbytes
    effective_gbs = logical_bytes / median_seconds / 1e9

    print(f"\n{name} inputs")
    print("Elements per input:", left.size)
    print("Input stride in bytes:", left.strides[0])
    print(f"Logical traffic: {logical_bytes / 1e6:.2f} MB")
    print(f"Median time: {median_seconds * 1e6:.2f} microseconds")
    print(f"Effective bandwidth: {effective_gbs:.2f} GB/s")


Contiguous inputs
Elements per input: 1000000
Input stride in bytes: 4
Logical traffic: 12.00 MB
Median time: 90.42 microseconds
Effective bandwidth: 132.72 GB/s

Strided inputs
Elements per input: 1000000
Input stride in bytes: 8
Logical traffic: 12.00 MB
Median time: 245.37 microseconds
Effective bandwidth: 48.90 GB/s


## 7. Independent work and dependency chains

Consider these independent calculations:

$$
z_i = x_i + 10.
$$

Computing z[3] does not require the result of z[0], z[1],
or z[2]. Once the inputs are available, these calculations
could be performed in any order.

Now consider a running total:

$$
s_i = s_{i-1} + x_i,
\qquad s_{-1} = 0.
$$

In this implementation, each addition needs the running total
produced by the previous addition. This creates a dependency chain.

Independent work gives a processor opportunities to overlap
operations. Dependencies can limit those opportunities.

Two useful terms are:

- Latency: how long an operation takes to produce its result.
- Throughput: how many operations finish per unit of time.

A processor can have high throughput by overlapping independent
operations, even when each individual operation takes several cycles.

The following Python example illustrates dependencies.
We are not using its timing to measure CPU instruction performance.

In [7]:
values = np.array([1, 2, 3, 4], dtype=np.float32)

independent = np.empty_like(values)
dependent = np.empty_like(values)

# Each output depends only on its own input.
for i in range(values.size):
    independent[i] = values[i] + 10.0

# Each step uses the running total produced by the previous step.
running_total = np.float32(0.0)

for i in range(values.size):
    running_total = running_total + values[i]
    dependent[i] = running_total

print("Inputs:", values)
print("Independent outputs:", independent)
print("Running totals:", dependent)

Inputs: [1. 2. 3. 4.]
Independent outputs: [11. 12. 13. 14.]
Running totals: [ 1.  3.  6. 10.]


## 8. Splitting work across CPU threads

A thread is a stream of execution within a program.
The operating system can schedule different threads on
different CPU cores.

Our elementwise additions are independent, so we can divide
the arrays into two halves:

- One task computes the first half.
- Another task computes the second half.

Both tasks read their input slices and write to separate,
non-overlapping slices of the output. No result from one task
is needed by the other.

We compare one direct NumPy call with two tasks submitted
to a pool of up to two worker threads.

Both implementations perform the same total number of additions.
Using more threads does not guarantee a speedup: submitting tasks
and coordinating their completion costs time, and the workers
share memory bandwidth.

We create the pool and warm up the operations before timing.
The threaded measurement includes task submission and waiting
for both tasks to finish.

In [8]:
from concurrent.futures import ThreadPoolExecutor

thread_output = np.empty_like(contiguous_x)
midpoint = thread_output.size // 2

# Create views of the two halves before timing.
chunks = [
    (contiguous_x[:midpoint], contiguous_y[:midpoint],
     thread_output[:midpoint]),
    (contiguous_x[midpoint:], contiguous_y[midpoint:],
     thread_output[midpoint:]),
]

def add_direct():
    np.add(contiguous_x, contiguous_y, out=thread_output)

with ThreadPoolExecutor(max_workers=2) as pool:

    def add_threaded():
        # Submit both tasks before waiting for either one.
        futures = [
            pool.submit(np.add, left, right, out=output)
            for left, right, output in chunks
        ]

        # Wait until both additions have finished.
        for future in futures:
            future.result()

    for name, operation in [
        ("Direct NumPy call", add_direct),
        ("Two-worker pool", add_threaded),
    ]:
        # Make unwritten output elements detectable.
        thread_output.fill(np.nan)

        for _ in range(5):
            operation()  # Untimed warmup.

        samples = []
        for _ in range(50):
            start = perf_counter()
            operation()
            end = perf_counter()
            samples.append(end - start)

        assert np.all(thread_output == 5.0)

        print(f"{name}: {median(samples) * 1e6:.2f} microseconds")

Direct NumPy call: 84.31 microseconds
Two-worker pool: 60.69 microseconds


## 9. Final observations

We measured CPU array addition and explored how execution overhead,
memory layout, dependencies, and threads affect performance.

### Recorded results

These measurements describe this machine and session.
Compare implementations within each experiment.

| Experiment | Elements | Median execution times |
|---|---:|---|
| Python loop vs NumPy | 100,000 | 8256.48 µs vs 6.63 µs |
| Contiguous vs strided inputs | 1,000,000 | 90.42 µs vs 245.37 µs |
| Direct NumPy call vs two-worker pool | 1,000,000 | 84.31 µs vs 60.69 µs |

### What I learned

- Define the timed region carefully. Preallocating the output
  excludes output allocation, while input reads, arithmetic,
  output writes, and call overhead remain included.

- Repeat measurements. The median summarizes typical observed
  timing and is less sensitive to occasional slow calls.

- NumPy avoids much of the per-element Python overhead by
  executing its array operations in compiled code. Our comparison
  does not isolate SIMD or prove that NumPy used multiple cores.

- Memory layout matters. Strided inputs contain gaps between
  selected elements, which can reduce useful data per fetched
  cache line. Their slower timing does not mean more additions.

- Logical traffic and hardware traffic are different quantities.
  Effective bandwidth calculated from logical bytes can reflect
  cache reuse rather than transfers from main memory.

- Dependencies constrain execution order. Independent additions
  can be divided among workers; our running-total loop requires
  the previous iteration's result.

- More threads do not guarantee proportional speedup. Our
  two-worker experiment achieved about 1.39× speedup, including
  task submission and completion-waiting overhead.

- Timing asynchronous tasks requires waiting for completion.
  Stopping the timer immediately after submission can
  underestimate the time needed to finish the computation.

### Next step

Move to an NVIDIA GPU and implement vector addition in Triton.
Learn how programs select their elements, handle array boundaries,
and produce results that we can check against a reference.